# Multi-Asset Options Tutorial

This tutorial demonstrates how to price multi-asset derivatives in QuantStrata:

1. **Basket Options** - Options on weighted portfolios
2. **Spread Options** - Options on price differences
3. **Rainbow Options** - Best-of and worst-of options

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Multi-Asset Simulation

First, let's understand how correlated asset simulation works.

In [ ]:
from src.models.multi_asset.simulation import (
    CorrelationMatrix,
    MultiAssetGBM,
)

# Create a 3-asset correlation structure
corr_matrix = np.array([
    [1.0, 0.6, 0.3],
    [0.6, 1.0, 0.5],
    [0.3, 0.5, 1.0]
])
correlation = CorrelationMatrix(corr_matrix)

print(f"Number of assets: {correlation.n_assets}")
print(f"\nCorrelation matrix:\n{correlation.matrix}")
print(f"\nCholesky factor L (where Σ = LL^T):\n{correlation.cholesky.round(3)}")

In [ ]:
# Create multi-asset GBM
gbm = MultiAssetGBM(
    spots=np.array([100.0, 100.0, 100.0]),
    r=0.05,
    dividends=np.array([0.02, 0.02, 0.02]),
    volatilities=np.array([0.2, 0.25, 0.3]),
    correlation=correlation,
)

# Simulate paths
sim = gbm.simulate(maturity=1.0, n_paths=5000, n_steps=252, seed=42)

print(f"Simulation shape: {sim.spots.shape}")
print(f"(paths, time steps + 1, assets)")

In [ ]:
# Visualize sample paths
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
time = np.linspace(0, 1, sim.n_steps + 1)

for i, ax in enumerate(axes):
    for path in range(5):  # Plot 5 sample paths
        ax.plot(time, sim.spots[path, :, i], alpha=0.7)
    ax.set_xlabel('Time (years)')
    ax.set_ylabel('Spot Price')
    ax.set_title(f'Asset {i+1} (σ={gbm.volatilities[i]:.0%})')
    ax.axhline(y=100, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.suptitle('Correlated GBM Paths', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Verify correlation structure in terminal values
terminals = sim.terminal_spots
log_returns = np.log(terminals / 100.0)

empirical_corr = np.corrcoef(log_returns.T)

print("Target Correlation:")
print(corr_matrix)
print("\nEmpirical Correlation:")
print(empirical_corr.round(3))

---

## 2. Basket Options

Basket options pay off based on a weighted portfolio of assets.

In [ ]:
from src.models.multi_asset.basket import (
    BasketParameters,
    basket_call_mc,
    basket_put_mc,
)

# Define basket parameters
basket_params = BasketParameters(
    spots=np.array([100.0, 100.0, 100.0]),
    weights=np.array([0.4, 0.35, 0.25]),  # Sum to 1
    strike=100.0,
    maturity=1.0,
    r=0.05,
    dividends=np.array([0.02, 0.02, 0.02]),
    volatilities=np.array([0.2, 0.25, 0.3]),
    correlation=correlation,
)

print(f"Basket spot value: {basket_params.basket_spot:.2f}")
print(f"Basket forward value: {basket_params.basket_forward:.2f}")

In [ ]:
# Price basket call and put
call_price, call_std = basket_call_mc(basket_params, n_paths=100000, seed=42)
put_price, put_std = basket_put_mc(basket_params, n_paths=100000, seed=42)

print(f"Basket Call: {call_price:.4f} ± {call_std:.4f}")
print(f"Basket Put:  {put_price:.4f} ± {put_std:.4f}")

# Verify put-call parity
discount = np.exp(-basket_params.r * basket_params.maturity)
parity = discount * (basket_params.basket_forward - basket_params.strike)
print(f"\nPut-Call Parity Check:")
print(f"  Call - Put = {call_price - put_price:.4f}")
print(f"  e^(-rT)(F-K) = {parity:.4f}")

### Correlation Effect on Basket Options

In [ ]:
# Study how correlation affects basket option prices
correlations = [0.0, 0.2, 0.4, 0.6, 0.8, 0.95]
call_prices = []
put_prices = []

for rho in correlations:
    corr = CorrelationMatrix.from_flat(rho, n=3)
    params = BasketParameters(
        spots=np.array([100.0, 100.0, 100.0]),
        weights=np.array([1/3, 1/3, 1/3]),
        strike=100.0, maturity=1.0, r=0.05,
        dividends=np.zeros(3),
        volatilities=np.array([0.3, 0.3, 0.3]),
        correlation=corr,
    )
    call, _ = basket_call_mc(params, n_paths=50000, seed=42)
    put, _ = basket_put_mc(params, n_paths=50000, seed=42)
    call_prices.append(call)
    put_prices.append(put)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(correlations, call_prices, 'b-o', linewidth=2)
axes[0].set_xlabel('Pairwise Correlation')
axes[0].set_ylabel('Basket Call Price')
axes[0].set_title('Basket Call vs Correlation')
axes[0].grid(True)

axes[1].plot(correlations, put_prices, 'r-s', linewidth=2)
axes[1].set_xlabel('Pairwise Correlation')
axes[1].set_ylabel('Basket Put Price')
axes[1].set_title('Basket Put vs Correlation')
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("Higher correlation → less diversification → higher basket volatility")

---

## 3. Spread Options

Spread options pay off based on the difference between two asset prices.

In [ ]:
from src.models.multi_asset.spread import (
    SpreadParameters,
    spread_call_mc,
    spread_put_mc,
    kirk_spread_call,
    kirk_spread_put,
    margrabe_exchange,
)

# Define spread parameters (e.g., crack spread)
spread_params = SpreadParameters(
    spot1=100.0,   # e.g., gasoline price
    spot2=95.0,    # e.g., crude oil price
    strike=5.0,    # spread strike
    maturity=0.5,
    r=0.05,
    q1=0.02,
    q2=0.01,
    sigma1=0.25,
    sigma2=0.30,
    rho=0.7,       # typically high for crack spreads
)

print(f"Current spread: {spread_params.spot1 - spread_params.spot2:.2f}")
print(f"Forward spread: {spread_params.spread_forward:.2f}")

In [ ]:
# Compare pricing methods
mc_call, mc_std = spread_call_mc(spread_params, n_paths=200000, seed=42)
kirk_call = kirk_spread_call(spread_params)

mc_put, _ = spread_put_mc(spread_params, n_paths=200000, seed=42)
kirk_put = kirk_spread_put(spread_params)

print("Spread Call Pricing:")
print(f"  Monte Carlo: {mc_call:.4f} ± {mc_std:.4f}")
print(f"  Kirk's Approx: {kirk_call:.4f}")
print(f"  Difference: {abs(mc_call - kirk_call):.4f}")

print("\nSpread Put Pricing:")
print(f"  Monte Carlo: {mc_put:.4f}")
print(f"  Kirk's Approx: {kirk_put:.4f}")

### Exchange Options (Margrabe's Formula)

In [ ]:
# When K=0, we have an exchange option with exact solution
exchange_params = SpreadParameters(
    spot1=100.0,
    spot2=100.0,
    strike=0.0,  # Exchange option
    maturity=1.0,
    r=0.05,
    q1=0.02,
    q2=0.02,
    sigma1=0.25,
    sigma2=0.30,
    rho=0.6,
)

# Exact Margrabe price
margrabe_price = margrabe_exchange(exchange_params)

# MC verification
mc_price, mc_std = spread_call_mc(exchange_params, n_paths=200000, seed=42)

print(f"Margrabe (exact): {margrabe_price:.4f}")
print(f"Monte Carlo: {mc_price:.4f} ± {mc_std:.4f}")

### Correlation Effect on Spread Options

In [ ]:
# Higher correlation = lower spread volatility = lower spread option value
rhos = np.linspace(-0.5, 0.95, 15)
spread_prices = []

for rho in rhos:
    params = SpreadParameters(
        spot1=100.0, spot2=100.0, strike=0.0, maturity=1.0,
        r=0.05, q1=0.02, q2=0.02, sigma1=0.3, sigma2=0.3, rho=rho
    )
    spread_prices.append(margrabe_exchange(params))

plt.figure(figsize=(10, 6))
plt.plot(rhos, spread_prices, 'g-o', linewidth=2)
plt.xlabel('Correlation (ρ)', fontsize=12)
plt.ylabel('Exchange Option Price', fontsize=12)
plt.title('Exchange Option Price vs Correlation\n(Higher ρ → assets move together → lower spread vol)', fontsize=14)
plt.grid(True)
plt.show()

---

## 4. Rainbow Options (Best-of / Worst-of)

Rainbow options have payoffs based on the best or worst performing asset.

In [ ]:
from src.models.multi_asset.rainbow import (
    RainbowParameters,
    best_of_call_mc,
    best_of_put_mc,
    worst_of_call_mc,
    worst_of_put_mc,
)

# Define rainbow parameters
rainbow_params = RainbowParameters(
    spots=np.array([100.0, 100.0, 100.0]),
    strike=100.0,
    maturity=1.0,
    r=0.05,
    dividends=np.array([0.02, 0.02, 0.02]),
    volatilities=np.array([0.2, 0.25, 0.3]),
    correlation=CorrelationMatrix.from_flat(0.5, n=3),
)

# Price all four types
best_call, _ = best_of_call_mc(rainbow_params, n_paths=100000, seed=42)
best_put, _ = best_of_put_mc(rainbow_params, n_paths=100000, seed=42)
worst_call, _ = worst_of_call_mc(rainbow_params, n_paths=100000, seed=42)
worst_put, _ = worst_of_put_mc(rainbow_params, n_paths=100000, seed=42)

print(f"Best-of Call:  {best_call:.4f}")
print(f"Best-of Put:   {best_put:.4f}")
print(f"Worst-of Call: {worst_call:.4f}")
print(f"Worst-of Put:  {worst_put:.4f}")

### Price Ordering

In [ ]:
# Compare to single-asset Black-Scholes
def bs_call(S, K, T, r, q, sigma):
    d1 = (np.log(S/K) + (r-q+0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*np.exp(-q*T)*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def bs_put(S, K, T, r, q, sigma):
    d1 = (np.log(S/K) + (r-q+0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return K*np.exp(-r*T)*norm.cdf(-d2) - S*np.exp(-q*T)*norm.cdf(-d1)

# Single asset calls/puts (using middle volatility)
single_call = bs_call(100, 100, 1.0, 0.05, 0.02, 0.25)
single_put = bs_put(100, 100, 1.0, 0.05, 0.02, 0.25)

print("Price Ordering for Calls:")
print(f"  Best-of:  {best_call:.4f}")
print(f"  Single:   {single_call:.4f}")
print(f"  Worst-of: {worst_call:.4f}")
print(f"  Ordering valid: {best_call > single_call > worst_call}")

print("\nPrice Ordering for Puts:")
print(f"  Worst-of: {worst_put:.4f}")
print(f"  Single:   {single_put:.4f}")
print(f"  Best-of:  {best_put:.4f}")
print(f"  Ordering valid: {worst_put > single_put > best_put}")

### Correlation Effects on Rainbow Options

In [ ]:
# Correlation has opposite effects on best-of vs worst-of
correlations = [0.0, 0.2, 0.4, 0.6, 0.8, 0.95]
best_calls = []
worst_calls = []

for rho in correlations:
    corr = CorrelationMatrix.from_flat(rho, n=3)
    params = RainbowParameters(
        spots=np.array([100.0, 100.0, 100.0]),
        strike=100.0, maturity=1.0, r=0.05,
        dividends=np.zeros(3),
        volatilities=np.array([0.3, 0.3, 0.3]),
        correlation=corr,
    )
    best, _ = best_of_call_mc(params, n_paths=50000, seed=42)
    worst, _ = worst_of_call_mc(params, n_paths=50000, seed=42)
    best_calls.append(best)
    worst_calls.append(worst)

plt.figure(figsize=(10, 6))
plt.plot(correlations, best_calls, 'b-o', linewidth=2, markersize=8, label='Best-of Call')
plt.plot(correlations, worst_calls, 'r-s', linewidth=2, markersize=8, label='Worst-of Call')
plt.axhline(y=single_call, color='gray', linestyle='--', label=f'Single-asset Call ({single_call:.2f})')
plt.xlabel('Correlation', fontsize=12)
plt.ylabel('Option Price', fontsize=12)
plt.title('Rainbow Option Prices vs Correlation', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True)
plt.show()

print("Key insight:")
print("- Higher correlation → Best-of converges DOWN to single-asset")
print("- Higher correlation → Worst-of converges UP to single-asset")

---

## 5. Practical Application: Structured Product

Let's price a reverse convertible note (investor sells worst-of put).

In [ ]:
# Reverse convertible on 3 stocks with 80% barrier
print("=" * 60)
print("Reverse Convertible Analysis")
print("=" * 60)
print("\nStructure: Investor sells worst-of put, receives coupon")
print("Barrier: 80% of initial (strike = 80)")
print("Underlyings: 3 correlated stocks")
print("Maturity: 1 year\n")

# Compare at different correlation levels
for rho in [0.3, 0.6, 0.9]:
    corr = CorrelationMatrix.from_flat(rho, n=3)
    params = RainbowParameters(
        spots=np.array([100.0, 100.0, 100.0]),
        strike=80.0,  # 80% barrier
        maturity=1.0,
        r=0.03,
        dividends=np.zeros(3),
        volatilities=np.array([0.25, 0.25, 0.25]),
        correlation=corr,
    )
    
    put_value, _ = worst_of_put_mc(params, n_paths=100000, seed=42)
    
    # Implied coupon (simplified: premium / notional)
    implied_coupon = put_value / 100 * 100  # as percentage
    
    print(f"Correlation ρ = {rho}:")
    print(f"  Worst-of Put Value: {put_value:.2f}")
    print(f"  Implied Coupon: ~{implied_coupon:.1f}%\n")

---

## Summary

| Product | Payoff | Correlation Effect |
|---------|--------|-------------------|
| Basket Call | max(Σ wᵢSᵢ - K, 0) | Higher ρ → Mixed |
| Spread Call | max(S₁ - S₂ - K, 0) | Higher ρ → Lower price |
| Best-of Call | max(max(Sᵢ) - K, 0) | Higher ρ → Lower price |
| Worst-of Call | max(min(Sᵢ) - K, 0) | Higher ρ → Higher price |

**Key takeaways:**
1. Correlation is crucial for multi-asset pricing
2. Spread options benefit from low correlation (more spread volatility)
3. Best-of benefits from low correlation (more dispersion)
4. Worst-of benefits from high correlation (less downside dispersion)
5. Kirk's approximation is fast but verify with MC for extreme parameters

---

*QuantStrata Tutorial | Phase 4.3 | January 2026*